In [ ]:
!pip install -q google-genai
!pip install -q transformers torch
!pip install -q neo4j sentence-transformers torch
!pip install -q neo4j sentence-transformers groq openai pandas tabulate
!pip install -q neo4j sentence-transformers google-genai
!pip install -q neo4j sentence-transformers groq google-genai
!pip install -q neo4j sentence-transformers google-genai transformers torch pandas accelerate
!pip install -q pandas


In [ ]:
!pip install -q groq neo4j sentence-transformers torch

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Load API keys from environment variables
GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is not configured. Please check your .env file.")

In [ ]:
import os
import time
import json
from groq import Groq
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# Step 1: Set Admin Password & Start Neo4j Server
# ---------------------------------------------------------
!./neo4j-community-5.18.0/bin/neo4j stop

# Set initial admin password for Neo4j 5.x
!./neo4j-community-5.18.0/bin/neo4j-admin dbms set-initial-password "password"

!./neo4j-community-5.18.0/bin/neo4j start
time.sleep(5)

# ---------------------------------------------------------
# Step 2: Initialize Driver with Basic Auth & Populate Data
# ---------------------------------------------------------
# GROQ_API_KEY is already loaded from environment variables (cell 3)
# No hardcoded API key here - using the one from .env file

# Authenticate with username 'neo4j' and password 'password'
driver = GraphDatabase.driver("bolt://127.0.0.1:7687", auth=("neo4j", "password"))
embedder = SentenceTransformer("all-MiniLM-L6-v2")
client = Groq(api_key=GROQ_API_KEY)

with driver.session() as session:
    session.run("""
    CREATE VECTOR INDEX `document_embeddings` IF NOT EXISTS
    FOR (d:Document) ON (d.embedding)
    OPTIONS {indexConfig: {`vector.dimensions`: 384, `vector.similarity_function`: 'cosine'}}
    """)

    docs = [
        {"id": "doc1", "text": "GraphRAG combines vector search with knowledge graph traversals to give LLMs structured context.", "cat": "Architecture"},
        {"id": "doc2", "text": "Neo4j Community Edition supports native HNSW vector indexes without enterprise licensing.", "cat": "Database"}
    ]

    for doc in docs:
        vec = embedder.encode(doc["text"]).tolist()
        session.run("""
        MERGE (d:Document {id: $id})
        SET d.text = $text, d.embedding = $vec
        MERGE (c:Category {name: $cat})
        MERGE (d)-[:BELONGS_TO]->(c)
        """, id=doc["id"], text=doc["text"], vec=vec, cat=doc["cat"])

print("Neo4j database authenticated, ready, and populated!")

# ---------------------------------------------------------
# Step 3: Define Agent Tools
# ---------------------------------------------------------
def vector_index_search(query: str) -> str:
    """Performs HNSW vector search to find matching document text in Neo4j."""
    try:
        query_vec = embedder.encode(query).tolist()
        with driver.session() as session:
            res = session.run("""
            CALL db.index.vector.queryNodes('document_embeddings', 2, $query_vec)
            YIELD node AS doc, score
            RETURN doc.text AS text, score
            """, query_vec=query_vec)
            results = [f"- {r['text']} (score: {round(r['score'], 2)})" for r in res]
            return "\n".join(results) if results else "No vector matches found."
    except Exception as e:
        return f"Database error: {str(e)}"

def graph_cypher_traversal(entity_name: str) -> str:
    """Searches connected entities and categories in Neo4j."""
    try:
        with driver.session() as session:

In [ ]:
# ---------------------------------------------------------
# 1. Self-Correcting Tool Definitions
# ---------------------------------------------------------

def graph_cypher_traversal(entity_name: str) -> str:
    """Searches connected entities and categories in Neo4j."""
    try:
        with driver.session() as session:
            res = session.run("""
            MATCH (d:Document)-[:BELONGS_TO]->(c:Category)
            WHERE toLower(d.text) CONTAINS toLower($entity) OR toLower(c.name) CONTAINS toLower($entity)
            RETURN d.text AS doc, c.name AS category
            """, entity=entity_name)
            results = [f"Doc: {r['doc']} | Category: {r['category']}" for r in res]

            # SELF-CORRECTION HINT: Guide the agent to try vector search if exact keyword match fails
            if results:
                return "\n".join(results)
            else:
                return f"NO GRAPH MATCHES for '{entity_name}'. FALLBACK REQUIRED: Call 'vector_index_search' with a broader query to perform semantic retrieval."
    except Exception as e:
        return f"Database error: {str(e)}"


# ---------------------------------------------------------
# 2. Agent Execution Loop with Fallback Guardrails
# ---------------------------------------------------------

def run_groq_agent_with_self_correction(user_goal: str, max_iterations: int = 4):
    print(f"User Goal: {user_goal}\n" + "="*50)

    # Enhanced system prompt instructing the agent on self-correction
    system_prompt = """
    You are an AI Agent with tool access. Follow these self-correction rules:
    1. If 'graph_cypher_traversal' returns no matches or a FALLBACK REQUIRED message, DO NOT give up or state that data is unavailable.
    2. Immediately call 'vector_index_search' using relevant semantic keywords as a secondary retrieval strategy.
    3. Only synthesize a final answer once all fallback options have been evaluated.
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_goal}
    ]

    for iteration in range(1, max_iterations + 1):
        print(f"\n[Iteration {iteration}] Agent Thinking...")

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=groq_tools,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls

        if tool_calls:
            messages.append(response_message)
            for tool_call in tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)
                print(f"Agent Action: Executing Tool `{fn_name}` with args: {fn_args}")

                if fn_name in TOOL_MAP:
                    tool_output = TOOL_MAP[fn_name](**fn_args)
                    print(f"Tool Output:\n{tool_output}")
                    messages.append({
                        "tool_call_id": tool_call.id,
                        "role": "tool",
                        "name": fn_name,
                        "content": tool_output
                    })
        else:
            print("\nFinal Answer Generated:")
            print("-" * 50)
            print(response_message.content.strip())
            return response_message.content.strip()

# Test the Self-Correcting Agent
run_groq_agent_with_self_correction("Search the knowledge graph for Neo4j features and calculate 15% of 248.85s latency.")

In [ ]:
import os
import json
import torch
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types

# 1. API Key Setup
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"  # Paste your active API key here

if GEMINI_API_KEY == "YOUR_GEMINI_API_KEY" or not GEMINI_API_KEY.strip():
    raise ValueError("Please replace 'YOUR_GEMINI_API_KEY' with your key from https://aistudio.google.com/")

# 2. Database & Model Setup
driver = GraphDatabase.driver("bolt://127.0.0.1:7687", auth=None)
embedder = SentenceTransformer("all-MiniLM-L6-v2")
client = genai.Client(api_key=GEMINI_API_KEY)

# 3. Define Tools
def vector_index_search(query: str) -> str:
    """Performs vector search to find document chunks in Neo4j."""
    query_vec = embedder.encode(query).tolist()
    with driver.session() as session:
        res = session.run("""
        CALL db.index.vector.queryNodes('document_embeddings', 2, $query_vec)
        YIELD node AS doc, score
        RETURN doc.text AS text, score
        """, query_vec=query_vec)
        results = [f"- {r['text']} (score: {round(r['score'], 2)})" for r in res]
        return "\n".join(results) if results else "No vector matches found."

def graph_cypher_traversal(entity_name: str) -> str:
    """Executes a graph query to find connected entities in Neo4j."""
    with driver.session() as session:
        res = session.run("""
        MATCH (d:Document)-[:BELONGS_TO]->(c:Category)
        WHERE d.text CONTAINS $entity OR c.name CONTAINS $entity
        RETURN d.text AS doc, c.name AS category
        """, entity=entity_name)
        results = [f"Doc: {r['doc']} | Category: {r['category']}" for r in res]
        return "\n".join(results) if results else "No graph relationship matches found."

def python_calculator(expression: str) -> str:
    """Evaluates mathematical expressions."""
    try:
        allowed_chars = "0123456789+-*/(). "
        if all(c in allowed_chars for c in expression):
            return str(eval(expression))
        return "Error: Invalid characters in expression."
    except Exception as e:
        return f"Execution error: {str(e)}"

TOOL_MAP = {
    "vector_index_search": vector_index_search,
    "graph_cypher_traversal": graph_cypher_traversal,
    "python_calculator": python_calculator
}

# 4. Agent Execution Loop
def run_agentic_graphrag(user_goal: str, max_iterations: int = 3):
    print(f"🎯 User Goal: {user_goal}\n" + "="*50)

    system_instruction = """
    You are an AI Agent with tool access.
    Solve the user's prompt by selecting tools, evaluating results, and returning a final answer.
    """

    messages = [f"User Goal: {user_goal}"]

    # Standard baseline model with function-calling support
    model_name = "gemini-1.5-flash"

    for iteration in range(1, max_iterations + 1):
        print(f"\n🔄 [Iteration {iteration}] Agent Thinking...")

        response = client.models.generate_content(
            model=model_name,
            contents="\n".join(messages),
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                tools=[vector_index_search, graph_cypher_traversal, python_calculator],
                temperature=0.1
            )
        )

        if response.function_calls:
            for call in response.function_calls:
                fn_name = call.name
                fn_args = call.args
                print(f"🛠️ Agent Action: Executing Tool `{fn_name}` with args: {fn_args}")

                if fn_name in TOOL_MAP:
                    tool_output = TOOL_MAP[fn_name](**fn_args)
                    print(f"📥 Tool Output:\n{tool_output}")
                    messages.append(f"Tool `{fn_name}` returned: {tool_output}")
        else:
            print("\n✅ Final Answer Generated:")
            print("-" * 50)
            print(response.text.strip())
            return response.text.strip()

# Run Agent
run_agentic_graphrag("Search the knowledge graph for Neo4j features and calculate 15% of 248.85s latency.")